# 2.1 - LBP + SVM (pré-processamento v2: MTCNN 224, margem 56)

Mesmo pipeline do `2.0` (mesmo `train_svm` e mesmo `evaluate`), lendo as imagens **v2** (`TRAIN_V2_DIR`, `VAL_V2_DIR`, `TEST_V2_DIR`),
mas com as características de `src/features_v2.py`, que seguem os artigos: LBP uniforme u2 de **59 bins** (P=8) e **243 bins** (P=16),
rosto reduzido para **64x64** dentro da extração (os arquivos em disco continuam 224) e Maatta com regiões sobrepostas de 14 px.
Dimensões: global 59, grid 531, Maatta 833. **Pré-requisito:** copiar `features_v2.py` para `src/` no repositório e subir para o GitHub.

**O que muda em relação ao 2.0**
- Entrada: `data/processed_v2_224/` (confere contagens do protocolo e tamanho 224 antes de rodar).
- Saída: só em `TCC/v2/` (características, seleção e log). O `experiment_log.csv` e a pasta `experiments/lbp_svm` do v1 **não são tocados**.
- Log próprio `TCC/v2/experiment_log_v2.csv`, com as mesmas colunas do log do v1 e com EER.
- A configuração é escolhida **automaticamente pela menor HTER na validação** e gravada em arquivo **antes** do teste.
- O teste só roda com `RUN_TEST = True`, usa a seleção gravada, e só é registrado uma vez.
- Nenhum arquivo de `src/` foi alterado: o log novo é uma função local deste notebook.

**Depois de reiniciar o runtime:** rode as células de 1 a 6 e depois a seção do teste (as características e a seleção já estão salvas).

In [ ]:
# Clona o repositório (ou reaproveita, se já existir) e instala as dependências.
# facenet-pytorch precisa de --no-deps (ver 2.0). Este notebook não roda MTCNN,
# mas src/dataset.py importa o facenet-pytorch.
import os
if not os.path.isdir('/content/liveness-detection'):
    !git clone https://github.com/laianemuckler/liveness-detection.git
%cd /content/liveness-detection
!git pull --quiet

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import csv
import json
from datetime import date, datetime

import numpy as np
import pandas as pd
from PIL import Image

from src.config import (
    TRAIN_V2_DIR, VAL_V2_DIR, TEST_V2_DIR,
    DRIVE_ROOT, LABEL_LIVE, LABEL_SPOOF,
)
from src.dataset import load_processed_split
from src.features_v2 import extract_features_v2, EXPECTED_DIMS_V2
from src.modeling.train import train_svm
from src.modeling.predict import evaluate

# ---------------- Saídas do v2 (pastas novas, nada do v1 é tocado) ----------------
OUTPUT_V2_ROOT = os.path.join(DRIVE_ROOT, 'v2')
RUNS_DIR = os.path.join(OUTPUT_V2_ROOT, 'experiments', 'lbp_svm', 'runs')
SELECTION_PATH = os.path.join(OUTPUT_V2_ROOT, 'experiments', 'lbp_svm', 'selection_lbp_v2.json')
LOG_V2_PATH = os.path.join(OUTPUT_V2_ROOT, 'experiment_log_v2.csv')
os.makedirs(RUNS_DIR, exist_ok=True)

RUN_TEST = False   # só mude para True no final, depois de a seleção estar gravada

# Contagens do protocolo (live, spoof) e dimensões esperadas de cada configuração (src/features.py)
EXPECTED_COUNTS = {'train': (1407, 1366), 'val': (336, 382), 'test': (3362, 5761)}
EXPECTED_DIMS = EXPECTED_DIMS_V2   # 59, 531 e 833 (LBP uniforme u2 de 59/243 bins, rosto em 64x64)

# id do experimento v2 -> configuração de características
CONFIGS_V2 = {
    'lbp_svm_v2_01_global': 'lbp_u2_global_p8r1',
    'lbp_svm_v2_02_grid': 'lbp_u2_grid_p8r1_3x3',
    'lbp_svm_v2_03_maatta': 'lbp_u2_maatta',
}

In [ ]:
# Log do v2: mesmas colunas e formatação de src.modeling.predict.log_experiment,
# mas gravando em TCC/v2/experiment_log_v2.csv (o log do v1 não é tocado).

def log_experiment_v2(exp_id, metodo, feature_config, modelo_config,
                      hter_val=None, auc_val=None, hter_test=None, auc_test=None,
                      eer_val=None, eer_test=None, obs=''):
    def fmt(v):
        return f"{v:.4f}" if v is not None else ''

    row = {
        'exp_id': exp_id, 'metodo': metodo, 'feature_config': feature_config,
        'modelo_config': modelo_config,
        'HTER_val': fmt(hter_val), 'AUC_val': fmt(auc_val), 'EER_val': fmt(eer_val),
        'HTER_test': fmt(hter_test), 'AUC_test': fmt(auc_test), 'EER_test': fmt(eer_test),
        'data': date.today().isoformat(), 'obs': obs,
    }
    file_exists = os.path.isfile(LOG_V2_PATH)
    os.makedirs(os.path.dirname(LOG_V2_PATH), exist_ok=True)
    with open(LOG_V2_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"Registrado '{exp_id}' em {LOG_V2_PATH}")


def ja_registrado(exp_id):
    """True se esse exp_id já está no log do v2 (evita duplicar linhas ao rodar de novo)."""
    if not os.path.isfile(LOG_V2_PATH):
        return False
    return exp_id in set(pd.read_csv(LOG_V2_PATH, dtype=str)['exp_id'])

In [ ]:
# Carrega as listas de imagens v2 e confere com o protocolo

train_paths, y_train = load_processed_split(TRAIN_V2_DIR)
val_paths, y_val = load_processed_split(VAL_V2_DIR)
test_paths, y_test = load_processed_split(TEST_V2_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")

for nome, y_, esperado in (('train', y_train, EXPECTED_COUNTS['train']),
                           ('val', y_val, EXPECTED_COUNTS['val']),
                           ('test', y_test, EXPECTED_COUNTS['test'])):
    y_ = np.array(y_)
    encontrado = (int(np.sum(y_ == LABEL_LIVE)), int(np.sum(y_ == LABEL_SPOOF)))
    print(f"  {nome:<5} live/spoof = {encontrado}")
    if encontrado != esperado:
        raise ValueError(f"Contagem de '{nome}' não bate com o protocolo: "
                         f"esperado {esperado}, encontrado {encontrado}.")

# garante que são as imagens v2 (224) e não as do v1 (160)
with Image.open(train_paths[0]) as im0:
    if im0.size != (224, 224):
        raise ValueError(f"Imagem com tamanho {im0.size}; o esperado no v2 é (224, 224).")
print("Imagens v2 conferidas (224x224).")

In [ ]:
# Funções do experimento (só definem; não rodam nada pesado)

def get_features(exp_dir, split, paths, feature_config_name):
    """Carrega X_<split>.npy se já existir (e confere o formato); senão extrai e salva.
    Nunca sobrescreve um arquivo existente."""
    npy_path = os.path.join(exp_dir, f'X_{split}.npy')
    esperado = (len(paths), EXPECTED_DIMS[feature_config_name])

    if os.path.exists(npy_path):
        X = np.load(npy_path)
        if X.shape != esperado:
            raise RuntimeError(f"{npy_path} tem formato {X.shape}, esperado {esperado}. "
                               f"Não vou sobrescrever: confira a pasta antes de continuar.")
        print(f"[{os.path.basename(exp_dir)}] X_{split}: carregado do disco {X.shape}")
        return X

    print(f"[{os.path.basename(exp_dir)}] extraindo '{feature_config_name}' ({split})...")
    X = extract_features_v2(paths, feature_config_name)
    if X.shape != esperado:
        raise ValueError(f"Formato inesperado {X.shape}; esperado {esperado}.")
    np.save(npy_path, X)
    return X


def run_lbp_experiment(exp_id, feature_config_name, svm_kernel='rbf', svm_C=1.0):
    """extrair (treino e validação) -> treinar SVM -> avaliar na validação -> registrar no log v2.
    As características do TESTE não são extraídas aqui."""
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    os.makedirs(exp_dir, exist_ok=True)

    X_train = get_features(exp_dir, 'train', train_paths, feature_config_name)
    X_val = get_features(exp_dir, 'val', val_paths, feature_config_name)

    print(f"[{exp_id}] treinando SVM...")
    model, scaler = train_svm(X_train, y_train, kernel=svm_kernel, C=svm_C)
    val_results = evaluate(model, scaler, X_val, y_val)
    print(f"[{exp_id}] Validação -> HTER: {val_results['HTER']*100:.2f}% | "
          f"AUC: {val_results['AUC']:.4f} | EER: {val_results['EER']*100:.2f}%")

    if ja_registrado(exp_id):
        print(f"[{exp_id}] já está no log do v2; não registrei de novo.")
    else:
        log_experiment_v2(
            exp_id=exp_id, metodo='LBP+SVM (v2 224)', feature_config=feature_config_name,
            modelo_config=f"kernel={svm_kernel}, C={svm_C}",
            hter_val=val_results['HTER'], auc_val=val_results['AUC'], eer_val=val_results['EER'],
            obs='pre-processamento v2: MTCNN 224, margem 56',
        )

    return {'feature_config': feature_config_name, 'svm_kernel': svm_kernel,
            'svm_C': svm_C, 'val_results': val_results}

In [ ]:
# Roda as três configurações (só treino e validação)

resultados = {}
for exp_id, feature_config_name in CONFIGS_V2.items():
    resultados[exp_id] = run_lbp_experiment(exp_id, feature_config_name)
    print()

print("Resumo da validação:")
for exp_id, exp in resultados.items():
    r = exp['val_results']
    print(f"  {exp_id}: HTER={r['HTER']*100:.2f}% | AUC={r['AUC']:.4f} | EER={r['EER']*100:.2f}%")

In [ ]:
# Seleção automática: menor HTER na validação. Gravada em arquivo ANTES de qualquer teste.
# Se o arquivo já existir, ele é mantido (a escolha não muda depois de ver o teste).

melhor_id = min(resultados, key=lambda k: resultados[k]['val_results']['HTER'])
print(f"Melhor na validação: {melhor_id} (HTER {resultados[melhor_id]['val_results']['HTER']*100:.2f}%)")

if os.path.exists(SELECTION_PATH):
    with open(SELECTION_PATH) as f:
        sel = json.load(f)
    print(f"Seleção já gravada: {sel['exp_id']}. Mantida.")
    if sel['exp_id'] != melhor_id:
        print("AVISO: a melhor desta rodada é diferente da seleção gravada.")
else:
    sel = {
        'exp_id': melhor_id,
        'feature_config': resultados[melhor_id]['feature_config'],
        'svm_kernel': resultados[melhor_id]['svm_kernel'],
        'svm_C': resultados[melhor_id]['svm_C'],
        'HTER_val': round(resultados[melhor_id]['val_results']['HTER'], 4),
        'criterio': 'menor HTER na validação',
        'gravado_em': datetime.now().isoformat(timespec='seconds'),
    }
    with open(SELECTION_PATH, 'w') as f:
        json.dump(sel, f, indent=1)
    print(f"Seleção gravada em {SELECTION_PATH}")

In [ ]:
# TESTE: só com RUN_TEST = True. Usa a seleção já gravada e só pode ser registrado uma vez.

if not RUN_TEST:
    print("RUN_TEST = False: o teste NÃO foi aberto. Mude para True só no final.")
else:
    if not os.path.exists(SELECTION_PATH):
        raise FileNotFoundError("Rode a seleção (célula 8) antes: ela ainda não foi gravada.")
    with open(SELECTION_PATH) as f:
        sel = json.load(f)

    exp_id = sel['exp_id']
    final_id = exp_id + '_FINAL_TEST'
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    if ja_registrado(final_id):
        raise RuntimeError(f"O teste de '{exp_id}' já foi registrado no log do v2. "
                           f"Ele é avaliado uma única vez.")

    print(f"Usando a seleção gravada: {exp_id} (HTER val {sel['HTER_val']*100:.2f}%)")

    # Retreina com as mesmas características e o mesmo seed (train_svm usa RANDOM_SEED)
    X_train = np.load(os.path.join(exp_dir, 'X_train.npy'))
    model, scaler = train_svm(X_train, y_train, kernel=sel['svm_kernel'], C=sel['svm_C'])

    X_test = get_features(exp_dir, 'test', test_paths, sel['feature_config'])
    test_results = evaluate(model, scaler, X_test, y_test)

    print(f"RESULTADO FINAL NO TESTE ({exp_id})")
    print(f"HTER: {test_results['HTER']*100:.2f}%")
    print(f"AUC:  {test_results['AUC']:.4f}")
    print(f"EER:  {test_results['EER']*100:.2f}%  (usa o melhor limiar do próprio teste, então é otimista)")

    log_experiment_v2(
        exp_id=final_id, metodo='LBP+SVM (v2 224)', feature_config=sel['feature_config'],
        modelo_config=f"kernel={sel['svm_kernel']}, C={sel['svm_C']}; escolhida pela validação",
        hter_test=test_results['HTER'], auc_test=test_results['AUC'], eer_test=test_results['EER'],
        obs='Métrica final de teste do LBP+SVM no v2',
    )

    pred_path = os.path.join(exp_dir, 'predictions_test.csv')
    if os.path.exists(pred_path):
        print(f"{pred_path} já existe; não sobrescrevi.")
    else:
        pd.DataFrame({
            'image_path': test_paths, 'label_true': y_test,
            'label_pred': test_results['y_pred'], 'score': test_results['y_scores'],
        }).to_csv(pred_path, index=False)
        print(f"Predições salvas em {pred_path}")

In [ ]:
# Log do v2 (só estes experimentos)
if os.path.isfile(LOG_V2_PATH):
    display(pd.read_csv(LOG_V2_PATH))
else:
    print("Ainda não há log do v2.")